# **Pre-Trained Model 1 - Efficient Net B0**

EfficientNetB0 was selected as the first pretrained architecture for the modelling phase. <br>


### **Why EfficientNet?**

EfficientNet is a CNN (convolutional neural network) built upon the concept of *compound scaling* — jointly scaling network depth, width and resolution via a fixed coefficient, rather than scaling any one dimension in isolation. This yields significantly better accuracy-per-parameter than other architectures.

### **Why B0 specifically?**

B0 is the baseline of the EfficientNet family (φ = 0). With only approximatelly 5M parameters it is the lightest variant, balacing **speed and accuracy**, making it ideal for a constrained medical dataset (~11,000 images). We believe that a heavier variant (B4–B7) would risk overfitting without substantially more data or aggressive regularisation. Therefore, B0 sits at the **optimal point on the bias–variance trade-off for this task**.

#### Moreover, we took into consideration even more specific dataset characteristics for these choice:
- **Suitability for dermoscopy**: Dermoscopic classification relies on fine-grained texture and colour pattern recognition (such as, atypical pigment networks, regression structures and vascular patterns). EfficientNet's depthwise separable convolutions and MBConv blocks are effective at capturing both local texture detail and global spatial context;
- **Transfer learning rationale**: ImageNet pre-training provides a rich, low-level feature initialisation (edges, textures, blobs) that are important for these medical images. Fine-tuning only the top layers in phase 1 prevents catastrophic forgetting of these representations and lets the new classification head converge stably.


### Imports

In [1]:
import os
import sys
import keras.applications
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
from keras.applications.efficientnet import preprocess_input


if os.getcwd().endswith('models'):
    os.chdir('..')
    
from utils.utils_model import *
from utils.utils_augmentation import *
from utils.utils_preproc import *

In [2]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus)
print("TF built with CUDA:", tf.test.is_built_with_cuda())
print("GPU available to TF:", tf.test.is_gpu_available())  # deprecated but still works

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TF built with CUDA: True
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
GPU available to TF: True


### **Data** Configuration

In [3]:
# Load label mapping first — everything else depends on it
with open("label2idx.json", "r") as f:
    label2idx = json.load(f)

N_CLASSES  = len(label2idx)
BATCH_SIZE = 32

train_df = pd.read_csv('data/augmented_metadata.csv')
val_df   = pd.read_csv('data/val_split.csv')
test_df  = pd.read_csv('data/test_split.csv')

# Normalise column names and encode labels for all splits
for df in [train_df, val_df, test_df]:
    if 'cleaned_path' in df.columns and 'image_path' in df.columns:
        df.drop(columns=['image_path'], inplace=True)
    df.rename(columns={'cleaned_path': 'image_path'}, inplace=True)
    df['dx_encoded'] = df['dx'].map(label2idx).astype(int)

# Build datasets
train_ds = make_dataset(train_df, shuffle=True, repeat=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)

STEPS_PER_EPOCH = len(train_df) // BATCH_SIZE
class_weights_dict = make_class_weights(train_df)

In [4]:
print(f"Steps per epoch: {STEPS_PER_EPOCH}")
print(f"Class weights: {class_weights_dict}")

Steps per epoch: 444
Class weights: {0: 1.6051948051948053, 1: 1.2503518648838845, 2: 0.7744360902255639, 3: 2.9861344537815127, 4: 0.7824938067712635, 5: 0.4512380952380952, 6: 2.188115763546798}


### **Model** Configuration

In [5]:
UNFREEZE_FROM_B = -30   # unfreeze last 30 layers of EfficientNetB0

def build_efficientnet():
    base = keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3),
        pooling=None
        
    )
    base.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))

    scaled_inputs = keras.layers.Rescaling(scale=255.0)(inputs)

    x = base(scaled_inputs, training=False)

    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.4)(x)
    x = keras.layers.Dense(256, activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(N_CLASSES)(x)
    return keras.Model(inputs, outputs, name="efficientnet_b0")

In [ ]:
# phase 1: head only
model_pt1 = build_efficientnet() # Model Pre-Trained 1
model_pt1.summary()

model_pt1.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history1_pt1 = model_pt1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=get_callbacks("checkpoints/model_b_phase1.weights.h5",
                            patience_es=6, patience_lr=3)
)

plot_history(history1_pt1, "Model B — EfficientNetB0 Phase 1 (head only)")

Model: "efficientnet_b0"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 rescaling_2 (Rescaling)     (None, 224, 224, 3)       0         
                                                                 
 efficientnetb0 (Functional)  (None, 7, 7, 1280)       4049571   
                                                                 
 global_average_pooling2d (G  (None, 1280)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 256)               327936    
                                                   

In [ ]:
# Phase 2: Fine-tune top layers
base_b = model_pt1.get_layer('efficientnetb0')
base_b.trainable = True

for layer in base_b.layers[:UNFREEZE_FROM_B]:
    layer.trainable = False

model_pt1.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy", , keras.metrics.AUC(name='auc')]
)

history2_b0 = model_pt1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    steps_per_epoch=STEPS_PER_EPOCH,
    class_weight=class_weights_dict,
    callbacks=get_callbacks("checkpoints/model_b_best.weights.h5",
                            patience_es=8, patience_lr=4)
)

plot_history(history2_pt1, "Model B — EfficientNetB0 Phase 2 (fine-tune)")
results_b = evaluate_model(model_pt1, test_ds, test_df, label2idx, "Model B — EfficientNetB0")
